In [1]:
!pip install -q datasets sentence-transformers faiss-cpu gradio transformers \
             torch peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 47.3 MB/s eta 0:00:00


In [2]:
import pandas as pd
import ast
import random
import json

# Load the four sports datasets
df1 = pd.read_parquet("/kaggle/input/datasets/ayaanfrngl/badminton-dataset")
df2 = pd.read_parquet("/kaggle/input/datasets/ayaanfrngl/basketball-dataset")
df3 = pd.read_parquet("/kaggle/input/datasets/ayaanfrngl/cricket-dataset")
df4 = pd.read_parquet("/kaggle/input/datasets/ayaanfrngl/football-dataset")

df = pd.concat([df1, df2, df3, df4], ignore_index=True)
print("Total raw rows:", len(df))

Total raw rows: 64685


In [3]:
# Parse QA pairs — FIX: use context from the row, not the answer
qa_pairs = []

for _, row in df.iterrows():
    try:
        ans = row["answer"]
        if isinstance(ans, str):
            ans = ast.literal_eval(ans)
        
        answer_text = ans["text"] if isinstance(ans, dict) else str(ans)
        
        # Skip empty answers
        if not answer_text or not str(answer_text).strip():
            continue

        qa_pairs.append({
            "question": str(row["question"]).strip(),
            "answer":   str(answer_text).strip(),
            "context":  str(row["context"]).strip()   # <-- actual context, not answer!
        })
    except Exception:
        continue

print("Valid QA pairs:", len(qa_pairs))

# Sample for manageable training time
random.seed(42)
qa_pairs = random.sample(qa_pairs, min(30_000, len(qa_pairs)))
print("After sampling:", len(qa_pairs))

Valid QA pairs: 41030
After sampling: 30000


In [4]:
#Sanity Check
for s in random.sample(qa_pairs, 3):
    print("Q:", s["question"])
    print("A:", s["answer"])
    print("CTX:", s["context"][:200])
    print("---")

Q: What club did Santoki play for for a number of years in England?
A: Farnham Royal cricket club
CTX: ty to deliver deceiving slower balls has made him an asset to T20 teams. Santokie spent a number of years playing in England for Farnham Royal cricket club where he used the English conditions to perf
---
Q: How many July 2015 events were held in Oman?
A: 25
CTX: 2 August 2008) Ireland (2 August 2008) Canada (2 August 2008) Bermuda (3 August 2008) Afghanistan (1 February 2010) Hong Kong (16 March 2014) Nepal (16 March 2014) United Arab Emirates (17 March 2014)
---
Q: What was the result of England's previous two Test series against England?
A: 5-0 victories
CTX: had been a hard-fought 1-1 draw at home against Pakistan, but their previous two Test series against England had both resulted in 5-0 victories. The first of these, in England in 1984, was the first w
---


In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# Lists for fast lookup
questions = [p["question"] for p in qa_pairs]
answers   = [p["answer"]   for p in qa_pairs]
contexts  = [p["context"]  for p in qa_pairs]

print("Encoding questions with sentence-transformers...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(questions, batch_size=128, show_progress_bar=True)
embeddings = np.array(embeddings, dtype="float32")

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"FAISS index built — {index.ntotal} vectors, dim={dimension}")

Encoding questions with sentence-transformers...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/235 [00:00<?, ?it/s]

FAISS index built — 30000 vectors, dim=384


In [6]:
# Test retrieval
def retrieve(query, k=3):                            #converting into vectors and finding 3 nearest neighbours
    q_emb = embedder.encode([query.lower().strip()])
    q_emb = np.array(q_emb, dtype="float32")
    D, I = index.search(q_emb, k=k)
    return D[0], I[0]

D, I = retrieve("Who won cricket world cup 2011?")
for dist, idx in zip(D, I):
    print(f"[dist={dist:.3f}] Q: {questions[idx]}")
    print(f"         A: {answers[idx]}")
    print()

[dist=0.016] Q: Who won the 2011 cricket world cup?
         A: India

[dist=0.237] Q: Who represented their country at the 2011 Cricket World Cup?
         A: cricketers

[dist=0.330] Q: How many countries participated in the 2011 Cricket World Cup?
         A: 14



In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, AutoConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
import torch

MODEL_ID = "microsoft/phi-2"

# 4-bit quantisation config (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,   # <-- change this line
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# FIX: patch pad_token_id onto config before model loads
print("Patching config...")
phi_config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
phi_config.pad_token_id = tokenizer.eos_token_id

print("Loading model in 4-bit...")
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    config=phi_config,          # <-- patched config
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
llm = prepare_model_for_kbit_training(llm)
print("Model loaded.")

Loading tokenizer...


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded.


In [8]:
# Fine tuning using Lora
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

llm = get_peft_model(llm, lora_config)
llm.print_trainable_parameters()   # should show ~0.5–1% of params

trainable params: 5,242,880 || all params: 2,784,926,720 || trainable%: 0.1883


In [9]:
#tokenising the prompt for fine tuning ahead
from datasets import Dataset

# Format each example as a prompt the model should learn to complete
def format_prompt(example):
    return (
        f"### Context:\n{example['context']}\n\n"
        f"### Question:\n{example['question']}\n\n"
        f"### Answer:\n{example['answer']}"
    )

def tokenize(example):
    text = format_prompt(example)
    enc = tokenizer(
        text,
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    enc["labels"] = enc["input_ids"].copy()
    return enc

TRAIN_SIZE = 8_000   
train_data = random.sample(qa_pairs, TRAIN_SIZE)
hf_dataset = Dataset.from_list(train_data)
tokenized_dataset = hf_dataset.map(tokenize, remove_columns=hf_dataset.column_names)

print("Train samples:", len(tokenized_dataset))

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Train samples: 8000


In [10]:
""" skip this as we have saved our trained data
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./phi2-sports-qa",
    num_train_epochs=1,
    per_device_train_batch_size=16,      # was 8
    gradient_accumulation_steps=1,       # was 2
    warmup_steps=20,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=llm,
    args=training_args,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer,
)

print("Starting fine-tuning...")
trainer.train()
print("Fine-tuning complete!")"""

' skip this as we have saved our trained data\nfrom transformers import TrainingArguments, DataCollatorForSeq2Seq\nfrom trl import SFTTrainer\n\ntraining_args = TrainingArguments(\n    output_dir="./phi2-sports-qa",\n    num_train_epochs=1,\n    per_device_train_batch_size=16,      # was 8\n    gradient_accumulation_steps=1,       # was 2\n    warmup_steps=20,\n    learning_rate=2e-4,\n    bf16=True,\n    logging_steps=10,\n    save_strategy="epoch",\n    optim="paged_adamw_8bit",\n    report_to="none",\n)\n\ntrainer = SFTTrainer(\n    model=llm,\n    args=training_args,\n    train_dataset=tokenized_dataset,\n    processing_class=tokenizer,\n)\n\nprint("Starting fine-tuning...")\ntrainer.train()\nprint("Fine-tuning complete!")'

In [11]:
from huggingface_hub import login
login()

In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, AutoConfig
from peft import PeftModel
import torch

MODEL_ID = "microsoft/phi-2"
ADAPTER_ID = "NocturnoCulto67/phi2-sports-qa"

# 🔹 Tokenizer
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 🔹 FIX: patch config BEFORE loading model
config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
config.pad_token_id = tokenizer.eos_token_id

# 🔹 Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 🔹 Load base model (with patched config)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    config=config,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True
)

# 🔹 Load LoRA
llm = PeftModel.from_pretrained(base_model, ADAPTER_ID)

llm.eval()

print("Model loaded successfully 🚀")

tokenizer_config.json:   0%|          | 0.00/318 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/969 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/10.5M [00:00<?, ?B/s]

Model loaded successfully 🚀


In [13]:
# Save the fine-tuned LoRA adapter
llm.save_pretrained("./phi2-sports-qa/final")

tokenizer.save_pretrained("./phi2-sports-qa/final")
print("Model saved.")

Model saved.


In [14]:
#RAG and chatbot mouth
llm.eval()

DISTANCE_THRESHOLD = 0.5
TOP_K = 3

def generate_answer(query: str, retrieved_context: str) -> str:
    prompt = (
        f"### Context:\n{retrieved_context}\n\n"
        f"### Question:\n{query}\n\n"
        f"### Answer:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(llm.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = llm.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only newly generated tokens
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # FIX: strip any leaked prompt template from the output
    for stop in ["### Question", "### Context", "### Answer", "Question:", "Answer:", "Context:"]:
        if stop in raw:
            raw = raw[:raw.index(stop)].strip()

    return raw if raw else "I could not generate an answer. Please try rephrasing."


def chatbot(query: str) -> str:
    if not query.strip():
        return "Please enter a question."

    D, I = retrieve(query, k=TOP_K)

    if D[0] > DISTANCE_THRESHOLD:
        return (
            "Sorry, I couldn't find relevant information for your query. "
            "Please ask something related to cricket, football, basketball, or badminton."
        )

    retrieved_context = "\n\n".join(
    [contexts[i] for i, d in zip(I, D) if d <= DISTANCE_THRESHOLD]
)

    return generate_answer(query, retrieved_context)


# Quick test
test_qs = [
    "Who won the Cricket World Cup in 2011?",
    "How many players are there in a basketball team?",
    "What is a shuttlecock used for?",
]

for q in test_qs:
    print("Q:", q)
    print("A:", chatbot(q))
    print()

Q: Who won the Cricket World Cup in 2011?
A: India

Q: How many players are there in a basketball team?
A: 30

Q: What is a shuttlecock used for?
A: Sorry, I couldn't find relevant information for your query. Please ask something related to cricket, football, basketball, or badminton.



In [15]:
import gradio as gr

def chat_fn(message, history):
    return chatbot(message)

demo = gr.ChatInterface(
    fn=chat_fn,
    title="⚽🏏 Sports QA Chatbot",
    description=(
        "Ask me anything about Cricket, Football, Basketball, or Badminton! "
        "Powered by Phi-2 (2.7B) fine-tuned with LoRA + FAISS retrieval."
    ),
    examples=[
        "Who won the FIFA World Cup in 2018?",
        "Who is considered the greatest cricketer of all time?",
        "How many players are in a basketball team on the court?",
        "What is the scoring system in badminton?",
    ],
)

demo.launch(share=True)  # share=True gives a public URL on Kaggle

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://dd217232d752f70707.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [16]:
D, I = retrieve("who won the fifa world cup in 2018?")
for i, d in zip(I, D):
    print(f"dist={d:.3f} | {contexts[i][:200]}")
    print()

dist=0.493 | in September 2018, after the 2018 FIFA World Cup, and Portugal won the tournament after defeating the Netherlands in the final in June 2019. The competition largely replaced international friendly fix

dist=0.498 | ions League and the UEFA Super Cup. Arrizabalaga won the 2012 European Championship with Spain's under-19 team. He made his senior debut in 2017, and was selected for the 2018 World Cup. Contents 1 Ho

dist=0.518 | r was Inverness Caledonian Thistle, who defeated Dumbarton in the 2018 final.



In [17]:
#from huggingface_hub import notebook_login
#notebook_login()

In [18]:
#llm.push_to_hub("NocturnoCulto67/phi2-sports-qa")
#tokenizer.push_to_hub("NocturnoCulto67/phi2-sports-qa")